# rustures Tutorial

This notebook walks through the complete workflow for finding **change points** in time-series data with `rustures`.

By the end, you will be able to:

- use `CostL2` to measure how homogeneous a segment is,
- use `Dynp` when the number of change points is known,
- use `Pelt` when the number of change points should be selected by a penalty,
- use `KernelCPD` to detect distributional changes beyond shifts in the mean,
- process two-dimensional signals with multiple features, and
- implement a Bernoulli custom cost in Python and connect it to `Dynp` and `Pelt`.

A breakpoint list returned by `rustures` always includes `n_samples` as its final element. For example, `[60, 120, 180]` represents the three segments `[0, 60)`, `[60, 120)`, and `[120, 180)`. The actual change points are 60 and 120.

## 0. Installation

Once `rustures` is available on PyPI, install it together with the plotting dependency used by this notebook:

```bash
python -m pip install rustures matplotlib
```

To test a wheel built from this repository instead, uncomment the next cell and replace the path with the wheel produced for your platform. The Python ABI and platform tags in the filename will vary by build target.

In [ ]:
# %pip install ../target/wheels/rustures-0.1.0-cp310-abi3-win_amd64.whl matplotlib

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import rustures

np.set_printoptions(precision=3, suppress=True)
print("rustures version:", rustures.__version__)

## 1. Create an Example Signal

The signal below contains 180 observations. It concatenates a segment with mean approximately 0, one with mean approximately 5, and one with mean approximately -3. The true change points are therefore 60 and 120. A small amount of Gaussian noise makes the signal resemble real measurements.

Change-point detection is not about finding a single unusual observation. It finds **boundaries where the statistical properties of a segment that was previously described well by one model change**. Here, the model describes every segment using one constant mean.

In [ ]:
rng = np.random.default_rng(42)
signal = np.r_[
    rng.normal(0.0, 0.25, 60),
    rng.normal(5.0, 0.25, 60),
    rng.normal(-3.0, 0.25, 60),
].astype(np.float64)
true_bkps = [60, 120, len(signal)]

print("shape:", signal.shape)
print("true breakpoints:", true_bkps)

In [ ]:
def plot_segmentation(values, bkps, title, true_bkps=None):
    """Plot a 1D/2D signal together with estimated change points."""
    values = np.asarray(values)
    if values.ndim == 1:
        values = values[:, None]

    fig, axes = plt.subplots(
        values.shape[1], 1,
        figsize=(11, 2.8 * values.shape[1]),
        sharex=True,
        squeeze=False,
    )
    for feature, ax in enumerate(axes[:, 0]):
        ax.plot(values[:, feature], color="#303030", linewidth=1)
        if true_bkps is not None:
            for bkp in true_bkps[:-1]:
                ax.axvline(bkp, color="#2ca02c", linestyle=":", linewidth=2, label="true")
        for bkp in bkps[:-1]:
            ax.axvline(bkp, color="#d62728", linestyle="--", linewidth=2, label="estimated")
        ax.set_ylabel(f"feature {feature}")
        handles, labels = ax.get_legend_handles_labels()
        unique = dict(zip(labels, handles))
        if unique:
            ax.legend(unique.values(), unique.keys(), loc="upper right")
    axes[-1, 0].set_xlabel("sample index")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

plot_segmentation(signal, [], "Synthetic piecewise-constant signal", true_bkps)

## 2. CostL2: How Well Can One Mean Describe a Segment?

`CostL2.error(start, end)` adds the squared deviations between every observation in `[start, end)` and the mean of that segment.

$$
C(s,e)=\sum_{t=s}^{e-1}(x_t-\bar{x}_{s:e})^2
$$

A segment that mixes data from different means has a large cost. Well-placed boundaries make the individual segments more homogeneous and reduce their total cost. Change-point detection can therefore be viewed as finding a partition that minimizes the following sum:

$$
C(0,b_1)+C(b_1,b_2)+\cdots+C(b_K,N)
$$

`fit` constructs prefix statistics for fast segment queries. Modifying the original array afterward does not change a cost object that has already been fitted.

In [ ]:
l2 = rustures.CostL2().fit(signal)

one_regime = l2.error(0, 60)
mixed_regimes = l2.error(0, 120)
manual_total = sum(l2.error(start, end) for start, end in zip([0] + true_bkps[:-1], true_bkps))
library_total = l2.sum_of_costs(true_bkps)

print(f"cost of [0, 60), one regime : {one_regime:.3f}")
print(f"cost of [0, 120), mixed      : {mixed_regimes:.3f}")
print(f"manual total cost            : {manual_total:.3f}")
print(f"sum_of_costs result           : {library_total:.3f}")
assert np.isclose(manual_total, library_total)

## 3. Dynp: When the Number of Change Points Is Known

`Dynp` fixes the number of change points, `n_bkps`, and uses dynamic programming to find the partition with the smallest total segment cost. Here, we provide the information that the signal contains two actual change points.

- `model="l2"`: describe each segment with a constant mean.
- `min_size=3`: require every segment to contain at least three samples.
- `jump=1`: consider every sample position as a boundary candidate. Increasing `jump` reduces the candidate count and can improve speed, but the result may become approximate.
- `n_bkps=2`: request two actual change points, excluding the terminal `n_samples` marker.

For every prefix, dynamic programming stores the minimum cost of dividing that prefix into an exact number of segments. When trying each possible start of the final segment, the optimal solution for everything before it has already been computed and can be reused. This produces a global optimum for a fixed number of change points without explicitly enumerating every possible partition.

In [ ]:
dynp = rustures.Dynp(model="l2", min_size=3, jump=1)
dynp_bkps = dynp.fit_predict(signal, n_bkps=2)

print("Dynp breakpoints:", dynp_bkps)
plot_segmentation(signal, dynp_bkps, "Dynp (L2, fixed K=2)", true_bkps)

## 4. Pelt: When the Number of Change Points Is Unknown

In real analyses, the number of change points is often not known beforehand. `Pelt` minimizes an objective that adds a penalty for every change point to the total segment cost.

$$
\text{objective}=\sum_{j}C(b_{j-1},b_j)+\beta\times(\text{number of changes})
$$

A small penalty $\beta$ favors many segments, while a large penalty favors a simpler partition. There is no universally correct value because an appropriate penalty depends on the data size and noise level. It is good practice to try several penalties and examine the stability of the number and locations of the detected changes.

With the L2 model, PELT safely prunes candidates that cannot become optimal in the future. It can therefore reduce the amount of computation substantially without changing the exact result.

In [ ]:
for pen in [1.0, 8.0, 1000.0]:
    bkps = rustures.Pelt(model="l2", min_size=3, jump=1).fit_predict(signal, pen=pen)
    print(f"pen={pen:7.1f} -> {bkps} (changes={len(bkps) - 1})")

pelt = rustures.Pelt(model="l2", min_size=3, jump=1).fit(signal)
pelt_bkps = pelt.predict(pen=8.0)
plot_segmentation(signal, pelt_bkps, "Pelt (L2, penalty=8)", true_bkps)

## 5. KernelCPD: Detect Changes Beyond the Mean

The L2 cost is particularly well suited to changes in the mean. `KernelCPD`, by contrast, maps observations into a feature space induced by a kernel and evaluates segment homogeneity there. An RBF kernel can respond not only to mean shifts but also to changes in variance or distributional shape.

The following example solves the same fixed-change-count problem with `kernel="rbf"`. The default high-performance `backend="fused"` combines kernel calculations with dynamic programming instead of precomputing a large Gram matrix and a complete table of segment costs.

Supported kernels are `linear`, `rbf`, and `cosine`. The appropriate choice depends on what should count as a homogeneous segment in your application.

In [ ]:
kernel_cpd = rustures.KernelCPD(
    kernel="rbf",
    min_size=3,
    jump=1,
    backend="fused",
)
kernel_bkps = kernel_cpd.fit_predict(signal, n_bkps=2)

print("KernelCPD breakpoints:", kernel_bkps)
print("resolved gamma:", kernel_cpd.gamma)
print("backend:", kernel_cpd.backend)
plot_segmentation(signal, kernel_bkps, "KernelCPD (RBF, fixed K=2)", true_bkps)

## 6. Signals with Multiple Features

An input with shape `(n_samples, n_features)` treats the features observed at the same time as one observation vector. The L2 cost sums squared distances from the segment mean across all features. It does not segment each feature independently; it finds **one shared breakpoint list for every feature**.

If features have very different units, the feature with the largest numerical range can dominate the cost. For real data, decide whether standardization is appropriate for the analysis before fitting the detector.

In [ ]:
rng_multi = np.random.default_rng(123)
multi_signal = np.column_stack([
    signal,
    np.r_[
        rng_multi.normal(2.0, 0.35, 60),
        rng_multi.normal(-2.0, 0.35, 60),
        rng_multi.normal(4.0, 0.35, 60),
    ],
]).astype(np.float64)

multi_bkps = rustures.Dynp(model="l2", min_size=3, jump=1).fit_predict(
    multi_signal, n_bkps=2
)
print("input shape:", multi_signal.shape)
print("multivariate breakpoints:", multi_bkps)
plot_segmentation(multi_signal, multi_bkps, "Multivariate Dynp (shared breakpoints)", true_bkps)

## 7. Python Custom Cost: Detect Changes in a Bernoulli Probability

When observations are either 0 or 1, a Bernoulli probability model is more natural than Gaussian mean-squared error. We estimate each segment's success probability with its sample mean and use the negative log-likelihood under that model as the segment cost.

A custom cost object follows this protocol:

- a positive integer attribute named `min_size`,
- `fit(signal)` to receive the signal and prepare the required statistics,
- `error(start, end) -> float` to return one finite segment cost, and
- optionally, `error_many(starts, ends) -> 1D float64 ndarray` to evaluate several segments at once.

The implementation below stores a prefix sum, so the number of ones in any segment can be retrieved quickly. A segment whose fitted probability is exactly 0 or 1 has cost zero, without numerically evaluating `log(0)`.

In [ ]:
class BatchBernoulliCost:
    min_size = 2

    def fit(self, signal):
        values = np.asarray(signal, dtype=np.float64)
        if values.ndim == 1:
            values = values[:, None]
        if not np.all((values == 0.0) | (values == 1.0)):
            raise ValueError("Bernoulli observations must be 0 or 1")

        zeros = np.zeros((1, values.shape[1]), dtype=np.float64)
        self.ones_prefix = np.vstack([zeros, np.cumsum(values, axis=0)])
        return self

    @staticmethod
    def _costs(ones, trials):
        mixed = (ones > 0.0) & (ones < trials)
        costs = np.zeros_like(ones, dtype=np.float64)
        probabilities = ones[mixed] / trials[mixed]
        costs[mixed] = (
            -ones[mixed] * np.log(probabilities)
            - (trials[mixed] - ones[mixed]) * np.log1p(-probabilities)
        )
        return costs

    def error(self, start, end):
        ones = self.ones_prefix[end] - self.ones_prefix[start]
        trials = np.full_like(ones, end - start, dtype=np.float64)
        return float(self._costs(ones, trials).sum())

    def error_many(self, starts, ends):
        starts = np.asarray(starts)
        ends = np.asarray(ends)
        ones = self.ones_prefix[ends] - self.ones_prefix[starts]
        lengths = (ends - starts)[:, None].astype(np.float64)
        trials = np.broadcast_to(lengths, ones.shape)
        return self._costs(ones, trials).sum(axis=1).astype(np.float64, copy=False)

In [ ]:
rng_binary = np.random.default_rng(7)
binary_signal = np.r_[
    rng_binary.binomial(1, 0.08, 60),
    rng_binary.binomial(1, 0.92, 60),
    rng_binary.binomial(1, 0.08, 60),
].astype(np.float64)
binary_true_bkps = [60, 120, len(binary_signal)]

bernoulli_dynp = rustures.Dynp(
    custom_cost=BatchBernoulliCost(), min_size=5, jump=1
)
binary_dynp_bkps = bernoulli_dynp.fit_predict(binary_signal, n_bkps=2)

bernoulli_pelt = rustures.Pelt(
    custom_cost=BatchBernoulliCost(), min_size=5, jump=1
)
binary_pelt_bkps = bernoulli_pelt.fit_predict(binary_signal, pen=8.0)

print("Dynp + custom Bernoulli:", binary_dynp_bkps)
print("Pelt + custom Bernoulli:", binary_pelt_bkps)
print("uses custom cost       :", bernoulli_dynp.uses_custom_cost)
print("uses batch callback    :", bernoulli_dynp.uses_batch_callback)
plot_segmentation(
    binary_signal, binary_dynp_bkps,
    "Dynp with a Python Bernoulli cost", binary_true_bkps
)

With a noisy finite sample, estimated change points may differ from the generating boundaries by a few positions. This is not necessarily an implementation error: for the observed binary sequence, the boundaries that minimize the objective can differ slightly from the locations where the population probabilities changed.

`error_many` does not change the algorithm's asymptotic time complexity, but it greatly reduces the number of calls between Rust and Python. If it is omitted, `error` provides a fallback with the same meaning, although Python callback overhead can become significant for larger inputs. The Python GIL is also held while a custom cost is in use. If maximum performance is required, a frequently used cost is better implemented as a native Rust cost.

Pelt with a custom cost uses exact, unpruned optimal partitioning because `rustures` does not assume that an arbitrary cost satisfies the inequality needed for safe pruning. It prioritizes correctness, but can therefore be slower than native L2 Pelt.

## 8. Choosing an API in Practice

| Situation | API to try first | Value to choose |
|---|---|---|
| The number of changes is known and mean shifts matter | `Dynp(model="l2")` | `n_bkps` |
| The number of changes is unknown and mean shifts matter | `Pelt(model="l2")` | `pen` |
| Variance or distributional shape changes also matter | `KernelCPD(kernel="rbf")` | `n_bkps` or `pen` |
| The problem needs a domain-specific statistical model | `Dynp/Pelt(custom_cost=...)` | cost definition and `n_bkps`/`pen` |

Use the following checklist for every detector:

1. Prepare the input as a finite, `float64`, one- or two-dimensional NumPy array.
2. Set `min_size` no smaller than the shortest segment that is meaningful for the application.
3. Use `jump=1` when exact candidate locations matter.
4. Remember that the final breakpoint is the signal endpoint `n_samples`, not an actual change point.
5. Do not rely on a single parameter setting; vary `n_bkps`, the penalty, or the kernel and examine result stability.
6. Always review detected changes using domain knowledge and a visualization of the original signal.

In [ ]:
summary = {
    "Dynp / L2": dynp_bkps,
    "Pelt / L2": pelt_bkps,
    "KernelCPD / RBF": kernel_bkps,
    "Dynp / multivariate L2": multi_bkps,
    "Dynp / custom Bernoulli": binary_dynp_bkps,
    "Pelt / custom Bernoulli": binary_pelt_bkps,
}
for name, bkps in summary.items():
    print(f"{name:27s} -> {bkps}")